In [1]:

# STEP 28: Prepare data for customer demand forecasting

import pandas as pd
import sqlite3
from pathlib import Path

# Find the project root
possible_roots = [Path("."), Path("..")]

project_root = next(
    (
        p for p in possible_roots
        if (p / "sql" / "06_bq4_monthly_demand.sql").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError("Project SQL folder not found")

# Connect to our existing SQLite database
db_path = (
    project_root / "data" / "processed"
    / "service_performance.db"
)

sql_path = (
    project_root / "sql" / "06_bq4_monthly_demand.sql"
)

with sqlite3.connect(db_path) as conn:
    monthly_data = pd.read_sql_query(
        sql_path.read_text(encoding="utf-8"),
        conn
    )

# Use complete calendar years for backtesting
forecast_data = monthly_data[
    monthly_data["Year"].between(2019, 2024)
].copy()

forecast_data["Date"] = pd.to_datetime(
    forecast_data["YearMonth"] + "-01"
)

forecast_data = forecast_data.sort_values("Date")

# Split into training and testing periods
train = forecast_data[
    forecast_data["Year"] <= 2023
].copy()

test = forecast_data[
    forecast_data["Year"] == 2024
].copy()

print("FORECASTING DATA PREPARATION")

print("Total months:", len(forecast_data))
print("Training months:", len(train))
print("Testing months:", len(test))

print("\nTRAINING DATA SAMPLE")
display(train.head())

print("\nTESTING DATA SAMPLE")
display(test.head())

# Validate the train-test split
assert len(train) == 60
assert len(test) == 12

assert train["Date"].max() < test["Date"].min()

print("\nFORECAST DATA VALIDATION PASSED!")


FORECASTING DATA PREPARATION
Total months: 72
Training months: 60
Testing months: 12

TRAINING DATA SAMPLE


,Year,Month,YearMonth,TotalCustomerVisits,RecordedCentreDays,RecordedDates,ReportingCentres,AvgVisitsPerCentreDay,Date
6,2019,1,2019-01,294082,1771,21,90,166.05,2019-01-01
7,2019,2,2019-02,272937,1757,20,90,155.34,2019-02-01
8,2019,3,2019-03,279809,1841,21,90,151.99,2019-03-01
9,2019,4,2019-04,262520,1657,19,90,158.43,2019-04-01
10,2019,5,2019-05,296061,1887,22,90,156.90,2019-05-01



TESTING DATA SAMPLE


,Year,Month,YearMonth,TotalCustomerVisits,RecordedCentreDays,RecordedDates,ReportingCentres,AvgVisitsPerCentreDay,Date
66,2024,1,2024-01,304001,1767,21,93,172.04,2024-01-01
67,2024,2,2024-02,298161,1899,23,93,157.01,2024-02-01
68,2024,3,2024-03,285914,1793,20,93,159.46,2024-03-01
69,2024,4,2024-04,290645,1809,21,93,160.67,2024-04-01
70,2024,5,2024-05,310697,1975,23,93,157.31,2024-05-01



FORECAST DATA VALIDATION PASSED!


In [2]:

# STEP 29: Seasonal Naive Forecast

# Use the corresponding months from 2023
seasonal_reference = train[
    train["Year"] == 2023
].set_index("Month")["AvgVisitsPerCentreDay"]

# Prepare the 2024 testing dataset
forecast_result = test[
    ["YearMonth", "Month", "AvgVisitsPerCentreDay"]
].copy()

# Predict each 2024 month using the same month in 2023
forecast_result["PredictedVisits"] = (
    forecast_result["Month"].map(seasonal_reference)
)

# Calculate the absolute forecasting error
forecast_result["AbsoluteError"] = (
    forecast_result["AvgVisitsPerCentreDay"]
    - forecast_result["PredictedVisits"]
).abs()

# Calculate absolute percentage error
forecast_result["APE"] = (
    forecast_result["AbsoluteError"]
    / forecast_result["AvgVisitsPerCentreDay"]
    * 100
)

# Calculate forecasting accuracy metrics
mae = forecast_result["AbsoluteError"].mean()
mape = forecast_result["APE"].mean()

wape = (
    forecast_result["AbsoluteError"].sum()
    / forecast_result["AvgVisitsPerCentreDay"].sum()
    * 100
)

# Validate results
assert forecast_result["PredictedVisits"].notna().all()
assert (forecast_result["AvgVisitsPerCentreDay"] > 0).all()

print("SEASONAL NAIVE FORECAST RESULTS")

display(
    forecast_result[
        [
            "YearMonth",
            "AvgVisitsPerCentreDay",
            "PredictedVisits",
            "AbsoluteError"
        ]
    ].round(2)
)

print("\nFORECAST EVALUATION")
print("Mean Absolute Error:", round(mae, 2))
print("Mean Absolute Percentage Error:", round(mape, 2), "%")
print("Weighted Absolute Percentage Error:", round(wape, 2), "%")


SEASONAL NAIVE FORECAST RESULTS


,YearMonth,AvgVisitsPerCentreDay,PredictedVisits,AbsoluteError
66,2024-01,172.04,177.23,5.19
67,2024-02,157.01,163.67,6.66
68,2024-03,159.46,161.42,1.96
69,2024-04,160.67,170.77,10.10
70,2024-05,157.31,161.08,3.77
71,2024-06,154.62,161.19,6.57
72,2024-07,153.99,161.57,7.58
73,2024-08,153.70,155.22,1.52
74,2024-09,151.58,154.32,2.74
75,2024-10,151.42,156.69,5.27



FORECAST EVALUATION
Mean Absolute Error: 4.83
Mean Absolute Percentage Error: 3.08 %
Weighted Absolute Percentage Error: 3.09 %


In [3]:

# STEP 30: Compare two forecasting models

# MODEL B: Historical monthly average (2019-2023)
monthly_average = train.groupby("Month")[
    "AvgVisitsPerCentreDay"
].mean()

# Add predictions from Model B
forecast_result["HistoricalAveragePrediction"] = (
    forecast_result["Month"].map(monthly_average)
)

# Actual observed demand
actual = forecast_result["AvgVisitsPerCentreDay"]

# Define a function to evaluate models
def evaluate_model(model_name, predictions):

    errors = actual - predictions

    mae = errors.abs().mean()

    mape = (
        errors.abs() / actual * 100
    ).mean()

    wape = (
        errors.abs().sum() / actual.sum() * 100
    )

    # Positive means the model overpredicts on average
    bias = (predictions - actual).mean()

    return {
        "Model": model_name,
        "MAE": round(mae, 2),
        "MAPE (%)": round(mape, 2),
        "WAPE (%)": round(wape, 2),
        "Bias": round(bias, 2)
    }


# Evaluate both models
comparison = pd.DataFrame([
    evaluate_model(
        "Seasonal Naive",
        forecast_result["PredictedVisits"]
    ),
    evaluate_model(
        "Historical Monthly Average",
        forecast_result["HistoricalAveragePrediction"]
    )
])

print("FORECASTING MODEL COMPARISON")
display(comparison)

# Check which model has the lowest MAE
best_model = comparison.loc[
    comparison["MAE"].idxmin(), "Model"
]

print("\nBest model by MAE:", best_model)


FORECASTING MODEL COMPARISON


,Model,MAE,MAPE (%),WAPE (%),Bias
0,Seasonal Naive,4.83,3.08,3.09,4.83
1,Historical Monthly Average,5.24,3.29,3.36,-3.93



Best model by MAE: Seasonal Naive


In [4]:

# STEP 31: Export forecasting results for Power BI

# Choose where to save the files
output_folder = project_root / "data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

# Prepare actual vs predicted results
forecast_export = forecast_result[
    [
        "YearMonth",
        "AvgVisitsPerCentreDay",
        "PredictedVisits",
        "HistoricalAveragePrediction"
    ]
].copy()

# Rename columns for easier reporting
forecast_export = forecast_export.rename(columns={
    "AvgVisitsPerCentreDay": "ActualVisits",
    "PredictedVisits": "SeasonalNaiveForecast",
    "HistoricalAveragePrediction": "HistoricalAverageForecast"
})

# Add a date column for Power BI
forecast_export["Date"] = pd.to_datetime(
    forecast_export["YearMonth"] + "-01"
)

# Calculate selected model's prediction error
forecast_export["ForecastError"] = (
    forecast_export["SeasonalNaiveForecast"]
    - forecast_export["ActualVisits"]
)

forecast_export["AbsoluteError"] = (
    forecast_export["ForecastError"].abs()
)

# Validate the export
assert len(forecast_export) == 12
assert forecast_export["ActualVisits"].notna().all()
assert forecast_export["SeasonalNaiveForecast"].notna().all()

# Save forecasts
forecast_export.to_csv(
    output_folder / "forecast_2024_validation.csv",
    index=False
)

# Save model comparison
comparison.to_csv(
    output_folder / "forecast_model_comparison.csv",
    index=False
)

print("FORECAST FILES EXPORTED SUCCESSFULLY!")

print("\nFiles saved:")
print("forecast_2024_validation.csv")
print("forecast_model_comparison.csv")

display(forecast_export.head())


FORECAST FILES EXPORTED SUCCESSFULLY!

Files saved:
forecast_2024_validation.csv
forecast_model_comparison.csv


,YearMonth,ActualVisits,SeasonalNaiveForecast,HistoricalAverageForecast,Date,ForecastError,AbsoluteError
66,2024-01,172.04,177.23,160.864,2024-01-01,5.19,5.19
67,2024-02,157.01,163.67,154.652,2024-02-01,6.66,6.66
68,2024-03,159.46,161.42,148.420,2024-03-01,1.96,1.96
69,2024-04,160.67,170.77,148.948,2024-04-01,10.10,10.10
70,2024-05,157.31,161.08,151.526,2024-05-01,3.77,3.77
